# BIL545 · 2. Hafta · Colab defteri: veri, künye ve uzaklık

Veri Madenciliğinde İleri Konular · Dr. Selma Güzel · Üsküdar Üniversitesi

Bu defterde: veriyi okumak ve saklama tiplerine bakmak, ilk **veri künyesi** tablosu, `describe()` ile beş sayı özeti ve 2. dersin **uzaklık ve benzerlik** hesaplarını kodla doğrulamak.

**Bu defter nasıl kullanılır?**

* Hücreleri **yukarıdan aşağıya, sırayla** çalıştırın: hücreye tıklayıp **Shift + Enter**. Sonraki hücreler öncekilerde tanımlanan değişkenleri kullanır; ortadan başlarsanız `NameError` hatası alırsınız. Defteri baştan çalıştırmak için: Çalışma zamanı → Tümünü çalıştır.
* Sunumdaki her kod, burada **satır satır ya da birkaç satırlık parçalar hâlinde** verilmiştir. Her parçanın önünde iki açıklama vardır:
  * 🐍 **Python:** satırdaki Python kavramı (Python bilmeniz gerekmez).
  * 📊 **Konu:** satırın veri madenciliği açısından ne yaptığı ve neden yapıldığı.
* 🔍 **Ara adım** hücreleri sunumda yoktur: bir hesabın neye uygulandığını görebilmeniz için ara sonucu gösterir (ör. ortalamanın hangi sütunun hangi değerlerine uygulandığı).
* Her bölümün sonunda **sunumdaki hâli** tek hücre olarak tekrar verilir; çıktısı sunumdakiyle aynı olmalıdır. ✅ **Sonucu okuyalım** kutusu sonucu yorumlar.
* Kodda `#` işaretinden sonraki her şey **yorumdur**: Python onu çalıştırmaz, insanlar için yazılmış nottur.

## 0. Sürümler

Her defterin ilk hücresinde kütüphane sürümlerini yazdırın (1. hafta defterindeki gibi). Projenizin README'sine de bunları yazacaksınız.

🐍 **Python:** `import` iki modülü açar; `__version__` her modülün sürüm numarasıdır.

In [ ]:
import sklearn, pandas
print("scikit-learn:", sklearn.__version__)
print("pandas      :", pandas.__version__)

## 1. Veriyi okumak ve saklama tiplerine bakmak

**Konu.** Analizin ilk adımı veriyi bir **tabloya** (pandas'ta *DataFrame*) okumak ve neye benzediğine bakmaktır: kaç satır (kayıt), kaç sütun (öznitelik) var, her sütun bilgisayarda hangi tiple saklanıyor?

**Veri: diabetes (Pima Indians Diabetes).** 768 kadın hasta; 8 ölçüm ve hedef sütunu `class` (diyabet testi pozitif mi, negatif mi). Sütunlar: `preg` gebelik sayısı, `plas` glikoz, `pres` kan basıncı, `skin` deri kıvrım kalınlığı, `insu` insülin, `mass` beden kitle indeksi, `pedi` diyabet soy ağacı fonksiyonu, `age` yaş. Weka'da açtığımız `diabetes.arff` ile aynı veridir; burada OpenML adlı açık veri deposundan indiriyoruz.

🐍 **Python:** pandas modülünü açar ve ona `pd` takma adını verir. Bundan sonra pandas'ın her aracı `pd.` ile başlar (ör. `pd.DataFrame`).

📊 **Konu:** pandas, veriyi satır ve sütunlardan oluşan bir tablo olarak tutar ve işler; Excel tablosunun Python'daki karşılığı gibi düşünebilirsiniz.

In [ ]:
import pandas as pd

🐍 **Python:** scikit-learn'ün `datasets` (veri kümeleri) bölümünden yalnız `fetch_openml` işlevini açar.

📊 **Konu:** `fetch_openml`, OpenML'deki bir veri kümesini adıyla internetten indirir.

In [ ]:
from sklearn.datasets import fetch_openml

🐍 **Python:** İlk iki satır `#` ile başladığı için **yorumdur**, çalışmaz. Kendi CSV dosyanızı okumak isterseniz baştaki `# ` işaretini silip dosya adını yazarsınız (dosyayı önce Colab'ın sol taraftaki Dosyalar bölmesine yüklemeniz gerekir).

Üçüncü satır **zincirleme** okunur: `fetch_openml(...)` bir sonuç nesnesi döndürür; onun `.frame` özelliği, verinin tablo hâlidir. Bu tablo `df` adına atanır (*data frame*'in kısaltması; ders boyunca tabloya hep bu adı vereceğiz).

📊 **Konu:** `version=1`: bu adla birden çok sürüm olabilir; hangisini istediğimizi sabitliyoruz (tekrarlanabilirlik). `as_frame=True`: sonucu pandas tablosu olarak ver.

In [ ]:
# Veriniz bir CSV dosyası olsaydı böyle okunurdu:
# df = pd.read_csv("veri.csv")
df = fetch_openml("diabetes", version=1, as_frame=True).frame

🐍 **Python:** `.shape` tablonun boyutudur: (satır sayısı, sütun sayısı). Parantezsiz yazılır, çünkü bir işlev değil, tablonun bir **özelliğidir**.

📊 **Konu:** 768 kayıt, 9 sütun (8 öznitelik ve hedef `class`).

In [ ]:
print(df.shape)

🐍 **Python:** `.dtypes` her sütunun **saklama tipini** listeler. `float64`: ondalıklı sayı; `int64`: tam sayı; `category`: sınırlı sayıda değeri olan kategorik sütun; `object`: genellikle metin.

📊 **Konu:** Dikkat: saklama tipi, 2. haftada gördüğümüz **ölçek tipini** (nominal, sıralı, aralık, oran) söylemez. `float64` bir sütun oran ölçekli bir ölçüm de olabilir, 0 ile kodlanmış eksik değerler içeren bir ölçüm de. Weka'daki *Type* alanıyla aynı sınır: tipi siz belirlersiniz.

In [ ]:
print(df.dtypes)

✅ **Sonucu okuyalım:** Tablo 768 × 9. Sekiz ölçümün tamamı `float64`, `class` ise `category` (iki değer: `tested_negative`, `tested_positive`). pandas "eksik yok" der, çünkü eksik değerler boş bırakılmamış, **0** yazılmış.

## 2. İlk künye tablosu

**Konu.** **Veri künyesi**, her öznitelik için temel kontrolleri tek tabloda toplar: saklama tipi, eksik değer oranı, farklı değer sayısı, sıfır oranı ve kimlik şüphesi. Weka'da her özniteliğe tek tek tıklayarak baktığımız bilgileri (Type, Missing, Distinct) tek hücrede verir ve Weka'da olmayan iki kontrol ekler. Proje 1'in 1. bölümündeki tip ve eksiklik tablosunun başlangıcıdır.

Bu bölümde birçok hesap "bir tablo üzerinde, **sütun sütun**" yapılıyor. Her adımda hesabın **neye** uygulandığını ara adımlarla göreceğiz.

🐍 **Python:** `select_dtypes("number")` tablodan yalnız **sayısal** sütunları seçer ve yeni bir tablo döndürür. Sonuç `sayisal` adına atanır; `df` değişmez.

📊 **Konu:** Sıfır oranı yalnız sayısal sütunlar için anlamlıdır. `class` kategorik olduğu için bu tabloya girmez.

In [ ]:
sayisal = df.select_dtypes("number")

🔍 **Ara adım:** `.columns` sütun adlarıdır; `.tolist()` onları düz bir listeye çevirir. İkinci listede `class` olmadığını görün.

In [ ]:
print(df.columns.tolist())
print(sayisal.columns.tolist())

Künyenin her sütununu ayrı ayrı hesaplayalım. Her hesap **bir sütun adı → bir sayı** eşlemesi üretir (pandas'ta buna *Series* denir: etiketli bir sayı listesi).

#### a) Saklama tipi

🔍 **Ara adım:** `df.dtypes` her sütunun tipini verir; `.astype(str)` bunları düz metne çevirir (tabloda düzgün görünsün diye).

In [ ]:
df.dtypes.astype(str)

#### b) Eksik oranı: `df.isna().mean()`

🔍 **Ara adım:** **1. adım:** `df.isna()` tablonun **her hücresi** için "bu hücre eksik (NaN) mi?" sorusunu sorar ve aynı boyutta bir **True/False tablosu** döndürür. İlk 5 satırını görüyoruz. Hepsi False: pandas'a göre hiç eksik yok.

In [ ]:
df.isna().head()

🔍 **Ara adım:** **2. adım (anlamak için):** `.sum()` her **sütundaki** True'ları sayar (True = 1, False = 0). Sonuç: her sütunda kaç eksik hücre var.

In [ ]:
df.isna().sum()

🔍 **Ara adım:** **3. adım:** `.mean()` her **sütunun** ortalamasını alır. True/False'ların ortalaması, True'ların **oranıdır**: 768 hücreden kaçının eksik olduğu ÷ 768. `.round(3)` üç basamağa yuvarlar. Yani `mean` burada tablonun değerlerine değil, **eksik mi? sorusunun cevaplarına** uygulanıyor.

In [ ]:
df.isna().mean().round(3)

#### c) Farklı değer sayısı: `df.nunique()`

🔍 **Ara adım:** `nunique()` (*number of unique*) her sütunda **kaç farklı değer** olduğunu sayar. Weka'daki *Distinct*. `class` için 2, `preg` için 17. Sürekli ölçümlerde (ör. `pedi`) bu sayı yüksektir.

In [ ]:
df.nunique()

#### d) Sıfır oranı: `(sayisal == 0).mean()`

🔍 **Ara adım:** **1. adım:** `sayisal == 0` her hücreyi 0 ile karşılaştırır; yine bir True/False tablosu (yalnız sayısal sütunlar).

In [ ]:
(sayisal == 0).head()

🔍 **Ara adım:** **2. adım:** her sütunun True oranı = o sütunda **0 olan değerlerin oranı**. Eksik oranıyla aynı mantık: `mean` sayıların kendisine değil, "sıfır mı?" cevaplarına uygulanıyor.

In [ ]:
(sayisal == 0).mean().round(3)

#### e) Hepsini bir tabloda birleştirmek

🐍 **Python:** `{ ... }` bir **sözlüktür** (dictionary): `anahtar: değer` çiftlerinden oluşur. Burada her anahtar, künyede bir **sütun adı**; her değer, yukarıda tek tek hesapladığımız sütun adı → sayı eşlemelerinden biri. `pd.DataFrame(sözlük)` bunları **etiketlerine göre** yan yana dizer: aynı sütun adının (ör. `insu`) dört değeri aynı satıra gelir.

Komut birden çok satıra yayılmış: süslü parantez kapanana kadar Python devam ettiğini bilir. Hizalamak için konan boşluklar sonucu etkilemez.

📊 **Konu:** Sonuç tablosunda **satırlar**, `df`'in sütunlarıdır (öznitelikler); **sütunlar**, her öznitelik için bir kontroldür. Veri tablosu ile künye tablosu bu yüzden birbirinin "yan yatmış" hâlidir.

In [ ]:
kunye = pd.DataFrame({
    "dtype":      df.dtypes.astype(str),          # saklama tipi (Weka: Type)
    "eksik_oran": df.isna().mean().round(3),      # NaN oranı (Weka: Missing)
    "farkli":     df.nunique(),                   # farklı değer sayısı (Weka: Distinct)
    "sifir_oran": (sayisal == 0).mean().round(3), # 0 olan değerlerin oranı
})

🐍 **Python:** `len(df)` tablonun satır sayısıdır (768). `kunye["farkli"] == len(df)` her satır için True/False üretir. Eşittirin solundaki `kunye["yeni_ad"] = ...` tabloya **yeni bir sütun ekler**.

📊 **Konu:** Bir sütundaki farklı değer sayısı satır sayısına eşitse, her kaydın değeri farklıdır: hasta no, sipariş no gibi bir **kimlik** olabilir. Kimlik sütunu modele girerse model kayıtları ezberler; 3. haftada ele alıyoruz.

In [ ]:
kunye["kimlik_suphesi"] = kunye["farkli"] == len(df)

🐍 **Python:** Tabloyu yazdırır.

📊 **Konu:** Satırlar alfabetik sıradadır (`age`, `class`, `insu`, …), çünkü dört eşleme aynı etiketleri taşımıyordu (sıfır oranında `class` yok); pandas birleştirirken etiketleri sıralar. `class` satırında `sifir_oran` **NaN** çıkar: o hesap yalnız sayısal sütunlarda yapıldı.

In [ ]:
print(kunye)

#### f) Sayısal özet: `describe()`

🐍 **Python:** `describe()` her **sayısal sütun** için sekiz özet hesaplar ve bunları satır satır yazar. `.T` (*transpose*) tabloyu çevirir: öznitelikler satıra, özetler sütuna geçer, okumak kolaylaşır.

📊 **Konu:** Sütunlar: `count` eksik olmayan değer sayısı, `mean` ortalama, `std` standart sapma, `min` en küçük, `25%` birinci çeyrek (Q1), `50%` **medyan**, `75%` üçüncü çeyrek (Q3), `max` en büyük. `min`, Q1, medyan, Q3, `max` birlikte 2. haftadaki **beş sayı özetidir**. Weka panelinde medyan ve çeyrekler yoktur.

In [ ]:
df.describe().T.round(2)   # 25% = Q1, 50% = medyan, 75% = Q3

✅ **Sonucu okuyalım:** `eksik_oran` her sütunda 0: pandas da Weka gibi "eksik yok" diyor. Ama `sifir_oran` `skin`'de yaklaşık 0,30, `insu`'da yaklaşık 0,49: eksik değerler **0 olarak kodlanmış**. `preg`'deki sıfırlar (0,145) ise gerçek: bu ayrımı tablo değil, alan bilgisi yapar. Kimlik şüphesi hiçbir sütunda yok.

`describe()` çıktısında `insu`'nun medyanı (30,5) ortalamasının (79,8) çok altında: sağa çarpıklık ve sıfır yığını birlikte. 3. haftada bu sıfırları önce NaN olarak işaretleyip sonra dolduracağız.

## 3. Uzaklık ve benzerlik hesaplarını doğrulamak

**Konu.** 2. haftada elle yaptığımız uzaklık ve benzerlik hesaplarını kodla tekrar ediyoruz; sayılar elle bulduklarınızla aynı olmalı. Üç örnek var:

1. **Sayısal vektörler:** `b`, `a`'nın tam 10 katı. Öklid, Manhattan, Chebyshev uzaklıkları ve kosinüs benzerliği.
2. **İkili (asimetrik) veri:** iki hastanın 10 taramalık sonucu. Jaccard ve basit eşleşme.
3. **Ölçekleme:** yaş ve gelir. Ölçeklemeden önce ve sonra Öklid uzaklığı.

🐍 **Python:** numpy modülünü `np` takma adıyla açar.

📊 **Konu:** numpy dizileriyle vektör hesabı yapacağız.

In [ ]:
import numpy as np

🐍 **Python:** scipy kütüphanesinin uzaklık bölümünden beş işlev açılıyor. Virgülle birden çok ad aynı satırda açılabilir.

📊 **Konu:** * `euclidean`: Öklid (L2) uzaklığı, √Σ(xᵢ − yᵢ)²
* `cityblock`: Manhattan (L1) uzaklığı, Σ|xᵢ − yᵢ|
* `chebyshev`: Chebyshev uzaklığı, en büyük |xᵢ − yᵢ|
* `cosine`: **kosinüs uzaklığı** = 1 − kosinüs benzerliği
* `jaccard`: **Jaccard uzaklığı** = 1 − Jaccard benzerliği

Dikkat: son ikisi benzerlik değil **uzaklık** döndürür. Benzerliği bulmak için 1'den çıkaracağız. Belgesini okumadan kullanılan bir işlev, istediğinizin tam tersini ölçüyor olabilir.

In [ ]:
from scipy.spatial.distance import euclidean, cityblock, chebyshev, cosine, jaccard

### 1. Sayısal vektörler

🐍 **Python:** `[2, 3, 0, 1]` bir Python **listesi**; `np.array(...)` onu bir numpy **dizisine** çevirir. Fark önemlidir: `10 * dizi` her elemanı 10 ile çarpar; `10 * liste` ise listeyi 10 kez arka arkaya ekler. Sayısal hesap için bu yüzden numpy dizisi kullanılır. Noktalı virgül (`;`) iki komutu tek satıra yazmaya yarar.

📊 **Konu:** `b`, `a` ile **aynı yönde** ama 10 kat büyük bir vektör.

In [ ]:
a = np.array([2, 3, 0, 1]); b = 10 * a

🔍 **Ara adım:** İlk satır iki diziyi gösterir: `b = [20 30 0 10]`. İkinci satır listeyle çarpmanın farkını gösterir: liste iki kez arka arkaya eklenir, sayılar çarpılmaz.

In [ ]:
print(a, b)
print([2, 3, 0, 1] * 2)

🔍 **Ara adım:** Öklid, Manhattan ve Chebyshev'i formülden elle hesaplayalım. `a - b` elemanları tek tek çıkarır. `** 2` kare alır, `.sum()` toplar, `np.sqrt` karekök alır: Öklid. `np.abs` mutlak değer: toplamı Manhattan, en büyüğü (`.max()`) Chebyshev.

In [ ]:
fark = a - b
print(fark)
print(np.sqrt((fark ** 2).sum()), np.abs(fark).sum(), np.abs(fark).max())

🐍 **Python:** Dört hesap tek `print` içinde, virgülle ayrılmış. `round(x, 3)` üç basamağa yuvarlar.

📊 **Konu:** `1 - cosine(a, b)`: scipy kosinüs **uzaklığı** verdiği için 1'den çıkarıp **benzerliği** buluyoruz.

In [ ]:
print(round(euclidean(a, b), 3), cityblock(a, b), chebyshev(a, b), round(1 - cosine(a, b), 3))

✅ **Sonucu okuyalım:** Öklid **33,675**, Manhattan **54**, Chebyshev **27** (en büyük fark: 3 ile 30), kosinüs benzerliği **1,0**. Öklid'e göre iki vektör uzak, kosinüse göre **özdeş** (aynı yön). Hangisi doğru? Soruya bağlı: büyüklük önemliyse Öklid, yalnız oranlar önemliyse kosinüs. Ölçüt seçimi bir modelleme kararıdır.

### 2. İkili veri: Jaccard ve basit eşleşme

🐍 **Python:** `dtype=bool`: dizinin elemanları sayı değil, **True/False** olarak saklanır (1 → True, 0 → False).

📊 **Konu:** On taramalık bir panelde iki hasta: 1 = pozitif bulgu, 0 = bulgu yok. Varlık bilgi taşır, yokluk çoğu zaman taşımaz: **asimetrik ikili** veri.

In [ ]:
x = np.array([1,0,1,0,0,0,0,1,0,0], dtype=bool)
y = np.array([1,0,0,0,0,0,0,1,0,1], dtype=bool)

🔍 **Ara adım:** Dört sayıyı elle sayalım. `&` "ve", `~` "değil" demektir; her ikisi de elemanlar üzerinde tek tek çalışır. q: ikisinde de 1 (tarama 1 ve 8), r: yalnız x'te 1, s: yalnız y'de 1, t: ikisinde de 0.

In [ ]:
q = (x & y).sum();  r = (x & ~y).sum();  s = (~x & y).sum();  t = (~x & ~y).sum()
print("q =", q, " r =", r, " s =", s, " t =", t)

🐍 **Python:** `x == y` her tarama için "iki hastanın sonucu aynı mı?" sorusunun True/False cevabıdır; `.mean()` True'ların **oranıdır**. Yine `1 - jaccard(...)` ile uzaklıktan benzerliğe geçiyoruz.

📊 **Konu:** Jaccard = q / (q + r + s) = 2 / 4; t'yi (ikisinde de bulgu yok) saymaz. Basit eşleşme = (q + t) / 10 = 8 / 10; t'yi sayar.

In [ ]:
print("Jaccard:", round(1 - jaccard(x, y), 3), " Basit eşleşme:", (x == y).mean())

✅ **Sonucu okuyalım:** Jaccard **0,5**, basit eşleşme **0,8**. Basit eşleşme, iki hastanın da bulgusu olmayan 6 taramayı "benzerlik" saydığı için şişiyor. Tarama verisi asimetrik olduğundan Jaccard uygundur. Panel 200 taramalık olsaydı basit eşleşme 1'e yaklaşırdı, Jaccard değişmezdi.

### 3. Ölçekleme: yaş ve gelir

🐍 **Python:** Çoklu atama: sağdaki üç dizi sırayla `A`, `B`, `C` adlarına atanır. Her dizi bir kişi: [yaş, gelir (TL)].

📊 **Konu:** A ile B yaşça yakın (25 ve 30), C ise çok farklı (60). Gelirleri birbirine yakın.

In [ ]:
A, B, C = np.array([25, 40000]), np.array([30, 42000]), np.array([60, 41000])

🐍 **Python:** İki Öklid uzaklığı, bir basamağa yuvarlanmış.

📊 **Konu:** Ölçeklemeden önce.

In [ ]:
print(round(euclidean(A, B), 1), round(euclidean(A, C), 1))

✅ **Sonucu okuyalım:** d(A, B) ≈ **2000**, d(A, C) ≈ **1000,6**. Öklid'e göre A'ya en yakın kişi **C**! Gelir farkı binlerle, yaş farkı onlarla ölçüldüğü için uzaklığı neredeyse tamamen gelir belirliyor: yaş hesapta yok gibi.

🐍 **Python:** İki dizi: `alt` her özniteliğin en küçük değeri, `genis` her özniteliğin **aralık genişliği**, yani maks − min.

📊 **Konu:** Yaş 18–70 aralığında: alt = 18, genişlik = 70 − 18 = **52**. Gelir 15 000–200 000 TL aralığında: alt = 15 000, genişlik = 200 000 − 15 000 = **185 000**. Dikkat: `genis` maksimum değil, maksimum ile minimumun **farkıdır**; bu yüzden formülde ayrıca `- alt` yazılmaz.

In [ ]:
alt, genis = np.array([18, 15000]), np.array([52, 185000])

🐍 **Python:** `def` ile kendi **işlevimizi** tanımlıyoruz: adı `minmaks`, aldığı değer (parametre) `p`. İki noktadan sonraki **girintili** satır işlevin gövdesidir; `return` sonucu geri verir. Python'da girinti zorunludur, bloğu o belirler. İşlev tanımlanınca hemen çalışmaz; çağrıldığında çalışır.

`(p - alt) / genis` numpy dizileriyle **eleman eleman** yapılır: yaştan 18 çıkarılıp 52'ye, gelirden 15 000 çıkarılıp 185 000'e bölünür.

📊 **Konu:** **Min-maks ölçekleme:** (x − min) / (maks − min). Her öznitelik [0, 1] aralığına gelir: en küçük değer 0, en büyük değer 1 olur. Böylece yaş ve gelir uzaklığa eşit ağırlıkla katılır.

In [ ]:
def minmaks(p):                  # min-maks ölçekleme: (x − min) / (maks − min)
    return (p - alt) / genis

🔍 **Ara adım:** Ölçeklenmiş değerler. A = (7/52, 25 000/185 000) ≈ (0,135; 0,135). İkinci satır kontrol: en küçük değerler (18, 15 000) 0'a, en büyük değerler (70, 200 000) 1'e gidiyor; ölçekleme doğru.

In [ ]:
print(minmaks(A), minmaks(B), minmaks(C))
print(minmaks(np.array([18, 15000])), minmaks(np.array([70, 200000])))

🐍 **Python:** İç içe çağrı, içten dışa: önce iki kişi ölçeklenir, sonra aralarındaki Öklid uzaklığı hesaplanır, sonra yuvarlanır.

📊 **Konu:** Ölçeklemeden sonra.

In [ ]:
print(round(euclidean(minmaks(A), minmaks(B)), 3), round(euclidean(minmaks(A), minmaks(C)), 3))

✅ **Sonucu okuyalım:** Ölçekleme sonrası d(A, B) ≈ **0,097**, d(A, C) ≈ **0,673**. Artık A'ya en yakın kişi **B**: sıralama tersine döndü. Ölçekleme tarafsız değildir; "her özniteliği eşit sayıyorum" demektir ve bu bir karardır. 3. haftada bir uyarı ekleyeceğiz: ölçekleyicinin min ve maks değerleri yalnız **eğitim** verisinden öğrenilmelidir, yoksa sızıntı olur.

**Sunumdaki hâli** (tek hücre):

In [ ]:
import numpy as np
from scipy.spatial.distance import euclidean, cityblock, chebyshev, cosine, jaccard

a = np.array([2, 3, 0, 1]); b = 10 * a
print(round(euclidean(a, b), 3), cityblock(a, b), chebyshev(a, b), round(1 - cosine(a, b), 3))

x = np.array([1,0,1,0,0,0,0,1,0,0], dtype=bool)
y = np.array([1,0,0,0,0,0,0,1,0,1], dtype=bool)
print("Jaccard:", round(1 - jaccard(x, y), 3), " Basit eşleşme:", (x == y).mean())

A, B, C = np.array([25, 40000]), np.array([30, 42000]), np.array([60, 41000])
print(round(euclidean(A, B), 1), round(euclidean(A, C), 1))
alt, genis = np.array([18, 15000]), np.array([52, 185000])
def minmaks(p):                  # min-maks ölçekleme: (x − min) / (maks − min)
    return (p - alt) / genis

print(round(euclidean(minmaks(A), minmaks(B)), 3), round(euclidean(minmaks(A), minmaks(C)), 3))